# downloads and installing

In [ ]:
# transformers torch
# dataset = https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences

# Loading data

In [ ]:
import pandas as pd


files = [
    '../Data/amazon_cells_labelled.txt',
     '../Data/imdb_labelled.txt',
        '../Data/yelp_labelled.txt'
]

data = []
for file in files:
    df = pd.read_csv(file, sep='\t', header=None, names=['sentence', 'label'])
    data.append(df)

# ترکیب همه دیتاست‌ها
df = pd.concat(data, ignore_index=True)
print(f"تعداد کل نمونه‌ها: {len(df)}")
print(df.head())

# Data preprocessing

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer

# tokenizer

In [ ]:
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer

# ساخت نمونه توکنایزر
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# تقسیم داده‌ها و ریست ایندکس
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# تابع توکنایز اصلاح شده
def tokenize_data(sentences, labels):
    input_ids = []
    attention_masks = []

    for sent in sentences:
        encoded_dict = tokenizer.encode_plus(
                            sent,
                            add_special_tokens=True,
                            max_length=64,
                            padding='max_length',
                            truncation=True,
                            return_attention_mask=True,
                            return_tensors='pt',
                       )
        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])

    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)
    labels = torch.tensor(labels.values)  # اصلاح اصلی

    return input_ids, attention_masks, labels

# توکنایز داده‌ها
train_inputs, train_masks, train_labels = tokenize_data(train_df['sentence'], train_df['label'])
test_inputs, test_masks, test_labels = tokenize_data(test_df['sentence'], test_df['label'])

# Loading the BERT model

In [ ]:
# وارد کردن کتابخانه‌ها
from transformers import BertForSequenceClassification, BertConfig, get_linear_schedule_with_warmup
from torch.optim import AdamW
import torch

# بارگذاری مدل
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels = 2,
    output_attentions = False,
    output_hidden_states = False,
)

# تنظیمات دستگاه
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# بهینه‌ساز و scheduler
optimizer = AdamW(model.parameters(), lr = 2e-5, eps = 1e-8)
epochs = 3
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0,
                                            num_training_steps = total_steps)

# حلقه آموزش
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_dataloader:
        b_input_ids, b_input_mask, b_labels = batch
        b_input_ids = b_input_ids.to(device)
        b_input_mask = b_input_mask.to(device)
        b_labels = b_labels.to(device)

        model.zero_grad()

        outputs = model(b_input_ids,
                        token_type_ids=None,
                        attention_mask=b_input_mask,
                        labels=b_labels)

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # اضافه کردن gradient clipping
        optimizer.step()
        scheduler.step()  # اضافه کردن scheduler

    avg_train_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_train_loss:.2f}")